In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostRegressor


TRAIN_PATH = Path("EDWaitTimes.csv")
TEST_PATH  = Path("EDWaitTimes.csv-test.csv")
DTR_PATH   = Path("dtr_a_18.csv")
OUT_PATH   = Path("predictions.csv")


PC_RE = re.compile(r"\b([ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z])\s?(\d[ABCEGHJ-NPRSTV-Z]\d)\b", re.I)

def normalize_name(name):
    s = "" if pd.isna(name) else str(name).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\.+$", "", s)
    return s

def extract_postal_code(address):
    if pd.isna(address): return np.nan
    m = PC_RE.search(str(address))
    return (m.group(1) + m.group(2)).upper() if m else np.nan

def extract_province(address):
    if pd.isna(address): return np.nan
    s = str(address).upper()
    m = re.search(r"\b(AB|BC|MB|NB|NL|NS|NT|NU|ON|PE|QC|SK|YT)\b", s)
    return m.group(1) if m else np.nan

def add_time_features(df):
    t = pd.to_datetime(df["time"], errors="coerce")
    df["dt"] = t
    df["hour"] = t.dt.hour
    df["dow"] = t.dt.dayofweek
    df["month"] = t.dt.month
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["dow_sin"] = np.sin(2*np.pi*df["dow"]/7)
    df["dow_cos"] = np.cos(2*np.pi*df["dow"]/7)
    return df

# load data
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

train["name"] = train["name"].astype(str).str.strip()
test["name"]  = test["name"].astype(str).str.strip()

# Hospital lookup from train data
meta = train[["name","address","latitude","longitude","open247"]].drop_duplicates("name").copy()
meta["name_norm"] = meta["name"].map(normalize_name)
meta["postalcode"] = meta["address"].map(extract_postal_code)
meta["province"]   = meta["address"].map(extract_province)

hospital_lookup = (
    meta.groupby("name_norm", as_index=False)
    .agg({
        "latitude":"median",
        "longitude":"median",
        "open247":lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
        "postalcode":lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
        "province":lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    })
)

# Load DTR Road Proximity Data 
# TODO: test how much improvement this gives in terms of score
dtr = pd.read_csv(DTR_PATH)
dtr["postalcode18"] = dtr["postalcode18"].astype(str).str.upper().str.replace(" ","")

road_cols = [c for c in dtr.columns if c.startswith("dtr")]

for c in road_cols:
    dtr[c] = pd.to_numeric(dtr[c], errors="coerce")
    dtr.loc[dtr[c].isin([-9999, -1111]), c] = np.nan

dtr["fsa"] = dtr["postalcode18"].str[:3]
dtr_fsa = dtr.groupby("fsa")[road_cols].mean().reset_index()
dtr_fsa.columns = ["fsa"] + [f"{c}_fsa_mean" for c in road_cols]

# Build features
train_start = pd.to_datetime(train["time"], errors="coerce").min()

def build_features(df):
    X = df[["name","time"]].copy()
    X["name_norm"] = X["name"].map(normalize_name)

    X = X.merge(hospital_lookup, on="name_norm", how="left")

    X["fsa"] = X["postalcode"].astype(str).str[:3]
    X["fsa"] = X["fsa"].replace("nan", "UNK")

    # join full postal code and FSA 
    X = X.merge(dtr[["postalcode18"]+road_cols],
                left_on="postalcode",
                right_on="postalcode18",
                how="left")
    X = X.merge(dtr_fsa, on="fsa", how="left")

    # Road features
    for c in road_cols:
        X[c] = X[c].fillna(X[f"{c}_fsa_mean"])
        X[c] = X[c].fillna(dtr[c].median())
        X[f"log1p_{c}"] = np.log1p(X[c])

    X = add_time_features(X)
    X["days_since_start"] = (
        (pd.to_datetime(X["time"]) - train_start).dt.total_seconds() / (3600*24)
    )

    X["open247_cat"] = X["open247"].map(
        lambda v: "True" if v is True else ("False" if v is False else "Unknown")
    )
    X["province_cat"] = X["province"].fillna("UNK")

    cat_cols = ["name_norm","fsa","open247_cat","province_cat"]

    for col in cat_cols:
        X[col] = X[col].fillna("UNK").astype(str)

    num_cols = [
        "latitude","longitude","hour","dow","month","is_weekend",
        "hour_sin","hour_cos","dow_sin","dow_cos",
        "days_since_start"
    ] + road_cols + [f"log1p_{c}" for c in road_cols]

    return X[cat_cols + num_cols]

X_train = build_features(train)
X_test  = build_features(test)

cat_cols = ["name_norm","fsa","open247_cat","province_cat"]
cat_idx  = [X_train.columns.get_loc(c) for c in cat_cols]

def train_target(y):
    y = pd.to_numeric(y, errors="coerce")
    mask = y.notna()
    X = X_train.loc[mask]
    y = y.loc[mask]

    model = CatBoostRegressor(
        iterations=2000,
        depth=8,
        learning_rate=0.05,
        loss_function="MAE",
        random_seed=42,
        verbose=200,
    )

    model.fit(X, np.log1p(y), cat_features=cat_idx)
    return model

wait_model = train_target(train["waitTimeMinutes"])
elos_model = train_target(train["elosMinutes"])

pred_wait = np.clip(np.expm1(wait_model.predict(X_test)), 0, None)
pred_elos = np.clip(np.expm1(elos_model.predict(X_test)), 0, None)

# save predictions
out = test[["name","time"]].copy()
out["waitTimeMinutes"] = np.round(pred_wait).astype(int)
out["elosMinutes"] = np.round(pred_elos).astype(int)

out.to_csv(OUT_PATH, index=False)

print("Predictions saved to:", OUT_PATH.resolve())
print(out.head(100))

0:	learn: 0.6218856	total: 68.3ms	remaining: 2m 16s
200:	learn: 0.2007612	total: 3.91s	remaining: 35s
400:	learn: 0.1875497	total: 7.9s	remaining: 31.5s
600:	learn: 0.1790907	total: 12.7s	remaining: 29.6s
800:	learn: 0.1728479	total: 16.6s	remaining: 24.9s
1000:	learn: 0.1682196	total: 21.2s	remaining: 21.1s
1200:	learn: 0.1643126	total: 25.3s	remaining: 16.8s
1400:	learn: 0.1607705	total: 29.5s	remaining: 12.6s
1600:	learn: 0.1580144	total: 33.7s	remaining: 8.39s
1800:	learn: 0.1551043	total: 37.8s	remaining: 4.17s
1999:	learn: 0.1527776	total: 41.9s	remaining: 0us
0:	learn: 0.4398130	total: 21.7ms	remaining: 43.5s
200:	learn: 0.0856784	total: 2.73s	remaining: 24.4s
400:	learn: 0.0787302	total: 5.32s	remaining: 21.2s
600:	learn: 0.0739927	total: 7.86s	remaining: 18.3s
800:	learn: 0.0707288	total: 10.4s	remaining: 15.6s
1000:	learn: 0.0679675	total: 13s	remaining: 13s
1200:	learn: 0.0657848	total: 15.5s	remaining: 10.3s
1400:	learn: 0.0637174	total: 18.1s	remaining: 7.74s
1600:	learn: 

ValueError: passed window 6H is not compatible with a datetimelike index